In [ ]:
import sys
sys.path.append("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline")
from config.constants import ICD_CHAPTERS
import matplotlib as mpl
mpl.rcParams["axes.grid"] = False


## Check the first D1 admission for each cohort

In [ ]:

# Load only the first admission for each patient

from pathlib import Path
import pandas as pd

MIMIC_IV_PATH = Path("/Users/zy51nise/Documents/BIONETs/FLabNet/Data/mimiciv/2.0/")
cohorts_dir = Path("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/DTB/")
labs_path = MIMIC_IV_PATH / "hosp" / "labevents.csv.gz"

# collect first hadm_id per patient across all cohorts
all_first = {}
for cohort_file in sorted(cohorts_dir.glob("cohort_*.csv.gz")):
    cohort_name = cohort_file.stem.replace("cohort_", "").replace(".csv", "")
    cohort = pd.read_csv(cohort_file, low_memory=False,usecols=["subject_id", "hadm_id", "admittime", "dischtime", "D1_date", "D2_date", "D1_W", "D1_5y"])
    cohort["admittime"] = pd.to_datetime(cohort["admittime"], format="%Y-%m-%d %H:%M:%S")
    cohort["dischtime"] = pd.to_datetime(cohort["dischtime"], format="%Y-%m-%d %H:%M:%S")
    first = (
        cohort.drop_duplicates("hadm_id")
        .sort_values("admittime")
        .groupby("subject_id")
        .first()
        .reset_index()[["subject_id", "hadm_id", "admittime", "dischtime", "D1_date", "D2_date", "D1_W", "D1_5y"]]
    )
    first["cohort"] = cohort_name
    all_first[cohort_name] = first

first_D1_df = pd.concat(all_first.values(), ignore_index=True)


#if (first_D1_df["admittime"].dt.date == pd.to_datetime(first_D1_df["D1_date"]).dt.date).all():
#    print("All D1 dates are same as admittime")

In [ ]:
# Load labs for subjects and < max dischtime
adm_info = first_D1_df[["subject_id", "hadm_id", "admittime", "dischtime"]].rename(columns={"hadm_id": "index_hadm_id"})
subject_max_disch = adm_info.groupby("subject_id")["dischtime"].max()
target_subs = set(subject_max_disch.index)

chunks = []
for chunk in pd.read_csv(
    labs_path, compression="gzip",
    usecols=["subject_id", "hadm_id", "charttime", "itemid", "valuenum"],
    chunksize=1_000_000,
):
    chunk["charttime"] = pd.to_datetime(chunk["charttime"], format="%Y-%m-%d %H:%M:%S")
    chunk = chunk[chunk["subject_id"].isin(target_subs)]
    if chunk.empty:
        continue
    chunk = chunk.merge(subject_max_disch.rename("max_dischtime"), on="subject_id")
    chunk = chunk[chunk["charttime"] <= chunk["max_dischtime"]]
    chunk = chunk.drop(columns=["max_dischtime"])
    chunks.append(chunk)

labs = pd.concat(chunks, ignore_index=True)

In [ ]:
#remove metadata
#metadata_labels = {
#    "Specimen Type",
#    "Estimated GFR (MDRD equation)",
#    "Length of Urine Collection",
#    "Green Top Hold (plasma)",
#    "I",
#    "L",
#    "H"
# "Estimated GFR (MDRD equation)","Light Green Top Hold", "Blue Top Hold", "Uhold"
#}
itemids_to_remove = [52033, 50947, 51087, 50933, 50934, 52026, 51678,50920,50887,50955,51103]
labs = labs[~labs['itemid'].isin(itemids_to_remove)]

### Extract features and labs for each first D1 admission

In [ ]:
from tqdm import tqdm
labs_by_subject = {sid: grp for sid, grp in labs.groupby("subject_id")}

unique_adm = first_D1_df.drop_duplicates("hadm_id")
all_features = []
itemid_counts = {}

for row in tqdm(unique_adm.itertuples(), total=len(unique_adm), desc="admissions"):
    admittime = row.admittime
    dischtime = row.dischtime
    
    adm_labs = labs_by_subject.get(row.subject_id)
    if adm_labs is None or adm_labs.empty:
        continue
    adm_labs = adm_labs[adm_labs["charttime"] <= dischtime]
    if adm_labs.empty:
        continue

    days_before = (dischtime - adm_labs["charttime"]).dt.total_seconds() / 86400

    features = {
        "hadm_id": row.hadm_id,
        "first_lab": adm_labs["charttime"].min(),
        "last_lab": adm_labs["charttime"].max(),
        "los_days": (dischtime - admittime).total_seconds() / 86400,
        "lod_days": (dischtime - adm_labs["charttime"].min()).total_seconds() / 86400,
        "pre_adm_days": max((admittime - adm_labs["charttime"].min()).total_seconds() / 86400, 0),
        "n_hadm_ids": adm_labs["hadm_id"].nunique(),
        "n_itemids": adm_labs["itemid"].nunique(),
        "n_lab_events": len(adm_labs),
        "n_lab_before_adm": (adm_labs["charttime"] < admittime).sum(),
        "n_lab_during_adm": (adm_labs["charttime"] >= admittime).sum(),
        #"lab_density_per_day": len(adm_labs) / max((dischtime - adm_labs["charttime"].min()).total_seconds() / 86400, 1),
    }
    hadm_itemids = {}
    for days in [15, 30, 90, 180, 365, 730]:
        mask = days_before <= days
        features[f"n_events_last_{days}d"] = mask.sum()
        features[f"n_itemids_last_{days}d"] = adm_labs.loc[mask, "itemid"].nunique()
        hadm_itemids[days] = adm_labs.loc[mask, "itemid"].value_counts().to_dict()


    itemid_counts[row.hadm_id] = hadm_itemids #itemid counts per admission and per window
    all_features.append(features)

features_df = pd.DataFrame(all_features)



-------------------
### Check all first D1 admissions together regardless of the cohort
-------------------

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle("Admission Lab Data Summary", fontsize=14, fontweight="bold")

def loghist(series, ax, title, xlabel):
    data = series.dropna()
    data = data[data > 0]
    bins = np.logspace(np.log10(data.min()), np.log10(data.max()), 50)
    ax.hist(data, bins=bins)
    ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("number of admissions")
    ax.grid(True, alpha=0.3)

loghist(features_df["los_days"], axes[0, 0], "Length of Stay", "days (log)")
loghist(features_df["lod_days"], axes[0, 1], "Length of Data", "days (log)")

# Prior admissions: clip at 20+
adm_counts = features_df["n_hadm_ids"].clip(upper=20)
adm_counts.value_counts().sort_index().plot.bar(ax=axes[0, 2])
axes[0, 2].set_title("Prior Admissions in Labs")
axes[0, 2].set_xlabel("number of prior admissions (20+ grouped)")
axes[0, 2].set_ylabel("number of subjects")
axes[0, 2].tick_params(axis="x", rotation=45)
axes[0, 2].grid(True, alpha=0.3, axis="y")

loghist(features_df["n_lab_events"], axes[1, 0], "Total Lab Events", "lab events (log)")
loghist(features_df["n_itemids"], axes[1, 1], "Unique Itemids", "unique itemids (log)")
#loghist(features_df["lab_density_per_day"], axes[1, 2], "Lab Density", "events/day (log)")

# absolute counts how many lab events we have in 30, 60,... days 
'''min_events = 100
coverage = [
    (features_df[f"n_events_last_{d}d"] >= min_events).mean() * 100
    for d in [15, 30, 90, 180, 365, 730]
]
axes[1, 2].plot([str(d) for d in [15, 30, 90, 180, 365, 730]], coverage, marker="o", color="steelblue")
axes[1, 2].set_title(f"% Admissions with ≥{min_events} Events in Last N Days")
axes[1, 2].set_xlabel("days before discharge")
axes[1, 2].set_ylabel("% of admissions")
axes[1, 2].set_ylim(50, 100)
axes[1, 2].grid(True, alpha=0.3)'''

# relative count, how many of all the measurements are taken within first 30,90 days
'''rel_coverage_median = [
    (features_df[f"n_events_last_{d}d"] / features_df["n_lab_events"]).median() * 100
    for d in [15, 30, 90, 180, 365, 730]
]
rel_coverage_p25 = [
    (features_df[f"n_events_last_{d}d"] / features_df["n_lab_events"]).quantile(0.25) * 100
    for d in [15, 30, 90, 180, 365, 730]
]
rel_coverage_p75 = [
    (features_df[f"n_events_last_{d}d"] / features_df["n_lab_events"]).quantile(0.75) * 100
    for d in [15, 30, 90, 180, 365, 730]
]
xlabels = [str(d) for d in [15, 30, 90, 180, 365, 730]]

axes[1, 2].plot(xlabels, rel_coverage_median, marker="o", color="steelblue", label="median")
axes[1, 2].fill_between(xlabels, rel_coverage_p25, rel_coverage_p75, alpha=0.3, color="steelblue", label="IQR")
axes[1, 2].set_title("% of Total Labs Captured in Last N Days")
axes[1, 2].set_xlabel("days before discharge")
axes[1, 2].set_ylabel("% of total lab events")
axes[1, 2].set_ylim(0, 100)
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)'''

windows = [15, 30, 90, 180, 365, 730]
thresholds = [50, 100, 200, 500]

for min_ev in thresholds:
    cov = [
        (features_df[f"n_events_last_{d}d"] >= min_ev).mean() * 100
        for d in windows
    ]
    axes[2, 0].plot([str(d) for d in windows], cov, marker="o", label=f"≥{min_ev} events")

axes[2, 0].set_title("% Admissions with Enough Data in Last N Days")
axes[2, 0].set_xlabel("days before discharge")
axes[2, 0].set_ylabel("% of admissions")
axes[2, 0].set_ylim(0, 110)
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)




hb = axes[1, 2].hexbin(
    features_df["n_lab_before_adm"].clip(lower=1),
    features_df["n_lab_during_adm"].clip(lower=1),
    gridsize=40, cmap="Blues", mincnt=1,
    xscale="log", yscale="log"
)
fig.colorbar(hb, ax=axes[1, 2], label="number of admissions")
axes[1, 2].set_xlabel("lab events before admission")
axes[1, 2].set_ylabel("lab events during admission")
axes[1, 2].set_title("Pre vs During Admission Labs")
axes[1, 2].grid(True, alpha=0.3)

# Coverage: median + IQR per window
windows = [15, 30, 90, 180, 365, 730]
medians = [features_df[f"n_events_last_{d}d"].median() for d in windows]
p25 = [features_df[f"n_events_last_{d}d"].quantile(0.25) for d in windows]
p75 = [features_df[f"n_events_last_{d}d"].quantile(0.75) for d in windows]
xlabels = [str(d) for d in windows]

axes[2, 1].plot(xlabels, medians, marker="o", color="steelblue", label="median")
axes[2, 1].fill_between(xlabels, p25, p75, alpha=0.3, color="steelblue", label="IQR")
axes[2, 1].set_title("Lab Events in Last N Days")
axes[2, 1].set_xlabel("days before discharge")
axes[2, 1].set_ylabel("number of lab events")
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

itemid_median = [features_df[f"n_itemids_last_{d}d"].median() for d in windows]
itemid_p25 = [features_df[f"n_itemids_last_{d}d"].quantile(0.25) for d in windows]
itemid_p75 = [features_df[f"n_itemids_last_{d}d"].quantile(0.75) for d in windows]

axes[2, 2].plot(xlabels, itemid_median, marker="o", color="darkorange", label="median")
axes[2, 2].fill_between(xlabels, itemid_p25, itemid_p75, alpha=0.3, color="darkorange", label="IQR")
axes[2, 2].set_title("Unique Itemids in Last N Days")
axes[2, 2].set_xlabel("days before discharge")
axes[2, 2].set_ylabel("number of unique itemids")
axes[2, 2].legend()
axes[2, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


-------------------
### Check first D1 admissions cohort based
-------------------

In [ ]:
features_with_cohort = features_df.merge(first_D1_df[["hadm_id", "cohort"]], on="hadm_id")


In [ ]:
def cohort_summary(g):
    row = {}
    for d in [15, 30, 90, 180, 365, 730]:
        row[f"cov_{d}d"] = (g[f"n_events_last_{d}d"] >= 50).mean() * 100
    row["median_los"] = g["los_days"].median()
    row["median_lod"] = g["lod_days"].median()
    row["median_n_events"] = g["n_lab_events"].median()
    row["median_n_itemids"] = g["n_itemids"].median()
    row["median_n_hadm_ids"] = g["n_hadm_ids"].median()
    row["median_pre_adm_days"] = g["pre_adm_days"].median()
    row["n_admissions"] = len(g)
    return pd.Series(row)

cohort_df = features_with_cohort.groupby("cohort").apply(cohort_summary).reset_index()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

fig = plt.figure(figsize=(16, 9))
fig.suptitle("Cohort-level Distribution of Key Metrics", fontsize=15, fontweight="bold", y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

metrics = [
    ("LOS (days)", "median_los", "#4C72B0"),
    ("Length of Data (days)", "median_lod", "#DD8452"),
    ("Total Lab Events", "median_n_events", "#55A868"),
    ("Unique Itemids", "median_n_itemids", "#C44E52"),
    ("Prior Admissions", "median_n_hadm_ids", "#8172B2"),
]

for i, (title, col, color) in enumerate(metrics):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    data = cohort_df[col].dropna()
    vp = ax.violinplot(data, positions=[0], widths=0.6, showmedians=False, showextrema=False)
    vp["bodies"][0].set_facecolor(color)
    vp["bodies"][0].set_alpha(0.4)
    ax.boxplot(data, positions=[0], widths=0.15, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color="white", linewidth=2),
               whiskerprops=dict(color=color),
               capprops=dict(color=color),
               flierprops=dict(marker="o", markerfacecolor=color, markersize=3, alpha=0.4))
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel("median per cohort", fontsize=9)
    ax.set_xticks([])
    median_val = data.median()
    ax.axhline(median_val, color=color, linestyle="--", linewidth=1, alpha=0.6)
    ax.annotate(f"median={median_val:.1f}", xy=(0.97, 0.05), xycoords="axes fraction",
                ha="right", fontsize=8, color=color)
    ax.grid(True, alpha=0.3, linewidth=0.5)

ax_cov = fig.add_subplot(gs[1, 2])
cov_cols = [f"cov_{d}d" for d in [15, 30, 90, 180, 365, 730]]
cov_data = cohort_df[cov_cols].rename(columns={f"cov_{d}d": f"{d}d" for d in [15, 30, 90, 180, 365, 730]})
bp = ax_cov.boxplot(cov_data.values, labels=cov_data.columns, patch_artist=True,
                    medianprops=dict(color="white", linewidth=2),
                    flierprops=dict(marker="o", markersize=3, alpha=0.4))
colors = sns.color_palette("Blues_d", len(bp["boxes"]))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax_cov.set_title("Coverage across cohorts", fontsize=11, fontweight="bold")
ax_cov.set_ylabel("% admissions with ≥50 events", fontsize=9)
ax_cov.set_xlabel("days before discharge", fontsize=9)
ax_cov.set_ylim(0, 110)
ax_cov.grid(True, alpha=0.3, linewidth=0.5)

plt.tight_layout()
plt.show()


In [ ]:

cohort_df["icd_chapter"] = cohort_df["cohort"].str[0].map(ICD_CHAPTERS)
metrics_to_compare = ["median_los", "median_lod", "median_n_events", "median_n_itemids", "cov_90d"]


In [ ]:
pivot = cohort_df.groupby("icd_chapter")[metrics_to_compare].median()
pivot_norm = (pivot - pivot.min()) / (pivot.max() - pivot.min())

plt.figure(figsize=(6, 5))
sns.heatmap(pivot_norm, annot=pivot.round(0), fmt="g", cmap="OrRd",
            linewidths=0.5, vmin=0, vmax=0.8,
            cbar_kws={"label": "value"},
            annot_kws={"size": 8})
plt.title("Cohort Metrics by Disease Group", fontsize=10)
plt.xlabel("")
plt.ylabel("")
plt.xticks(fontsize=8, rotation=30, ha="right")
plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.show() 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, len(metrics_to_compare), figsize=(18, 8), sharey=True)

for ax, col in zip(axes, metrics_to_compare):
    data = cohort_df.copy()
    chapter_order = data.groupby("icd_chapter")[col].median().sort_values().index
    sns.stripplot(data=data, y="icd_chapter", x=col, ax=ax, order=chapter_order,
                  size=3, alpha=0.5, jitter=True)
    sns.pointplot(data=data, y="icd_chapter", x=col, ax=ax, order=chapter_order,
                  color="black", markers="D", markersize=5, linewidth=1, errorbar=None)
    ax.set_title(col.replace("median_", "").replace("_", " "), fontsize=18)
    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_xlabel("")
    #if ax != axes[0]:
    ax.set_ylabel("")

plt.suptitle("Metric distributions by disease chapter", fontweight="bold", fontsize=22)
plt.tight_layout()
plt.show()

In [ ]:
# big picture

sort_by = "median_lod" 
sort_ascending = False

chapter_colors = {
    "A": "#e41a1c", "B": "#e41a1c",
    "C": "#ff7f00", "D": "#ffb300",
    "E": "#4daf4a",
    "F": "#984ea3",
    "G": "#377eb8",
    "H": "#a65628",
    "I": "#f781bf",
    "J": "#17becf",
    "K": "#8c564b",
    "L": "#bcbd22",
    "M": "#2ca02c",
    "N": "#9467bd",
    "O": "#e377c2",
    "Q": "#7f7f7f",
    "R": "#1f77b4",
    "S": "#d62728", "T": "#d62728",
    "Z": "#aec7e8",
}

icd_labels = pd.read_csv(MIMIC_IV_PATH / "hosp" / "d_icd_diagnoses.csv.gz", usecols=["icd_code", "icd_version", "long_title"])
icd10_3digit = icd_labels[icd_labels["icd_version"] == 10].copy()
icd10_3digit["icd_3"] = icd10_3digit["icd_code"].str[:3]
icd10_3digit = icd10_3digit.drop_duplicates("icd_3").set_index("icd_3")["long_title"]

cohort_df["icd_start"] = cohort_df["cohort"].str.extract(r"^([A-Z]\d{2})")
cohort_df["icd_label"] = cohort_df["icd_start"] + " - " + cohort_df["icd_start"].map(icd10_3digit).fillna("Unknown")

cohort_counts = cohort_df.groupby("icd_label").size().rename("n_cohorts")
pivot = cohort_df.groupby("icd_label")[metrics_to_compare].median().join(cohort_counts)
#pivot = pivot.sort_values(sort_by, ascending=sort_ascending)

display_cols = metrics_to_compare + ["n_cohorts"]
pivot_norm = (pivot[display_cols] - pivot[display_cols].min()) / (pivot[display_cols].max() - pivot[display_cols].min())

fig, ax = plt.subplots(figsize=(16, len(pivot) * 0.35))
sns.heatmap(pivot_norm, annot=pivot[display_cols].round(0), fmt="g", cmap="GnBu",
            linewidths=0.5, vmin=0, vmax=0.8,
            annot_kws={"size": 7}, ax=ax)
ax.set_title(f"All Cohorts by ICD Code - Key Metrics (sorted by {sort_by})", fontsize=11, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=8, rotation=30, ha="right")

for label in ax.get_yticklabels():
    icd_code = label.get_text().split(" - ")[0].strip()
    color = chapter_colors.get(icd_code[0], "black")
    label.set_color(color)
    label.set_fontsize(12)

plt.tight_layout()
plt.subplots_adjust(left=0.45)
plt.show()


## Features selection analysis

In [ ]:

adm_chapter = first_D1_df[["subject_id","hadm_id", "cohort"]].copy()
adm_chapter["icd_chapter"] = adm_chapter["cohort"].str[0].map(ICD_CHAPTERS)
hadm_to_chapters = adm_chapter.groupby("hadm_id")["icd_chapter"].apply(set).to_dict()

# Conver itemid counts dict to Df
rows = []
for hadm_id, windows in itemid_counts.items():
    for days, counts in windows.items():
        for itemid, count in counts.items():
            rows.append({"hadm_id": hadm_id, "window": days, "itemid": itemid, "count": count})

lab_counts = pd.DataFrame(rows) # hadm_id, window, itemid, count

lab_items = pd.read_csv(MIMIC_IV_PATH / "hosp" / "d_labitems.csv.gz", usecols=["itemid", "label"])
itemid_label = lab_items.set_index("itemid")["label"]

In [ ]:
#select window to include only the labs in that specific window
window = 90
lab_counts_per_window = lab_counts[lab_counts.window == window].drop(columns ="window")


In [ ]:
# Prevalence of itemids in each chapter

from collections import defaultdict

chapter_itemid_count = defaultdict(lambda: defaultdict(int))
chapter_size = defaultdict(int)
seen_hadm = defaultdict(set)


for row in lab_counts_per_window.itertuples(index=False):
    chapters = hadm_to_chapters.get(row.hadm_id, set())
    for chapter in chapters:
        if row.hadm_id not in seen_hadm[chapter]:
            chapter_size[chapter] += 1
            seen_hadm[chapter].add(row.hadm_id)
        chapter_itemid_count[chapter][row.itemid] += row.count

    
# sum of ALL measurements across ALL itemids in the chapter
# chapter_total = {chapter: sum(counts.values()) for chapter, counts in chapter_itemid_count.items()}

all_chapters = sorted(chapter_size.keys())

# on average how many times was this itemid measured per admission in this chapter?
prevalence = {}
for chapter in all_chapters:
    prevalence[chapter] = {}
    for itemid, count in chapter_itemid_count[chapter].items():
        prevalence[chapter][itemid] = count / chapter_size[chapter]

In [ ]:
# How similar are the chapters?

chapters = sorted(seen_hadm.keys())
n = len(chapters)
overlap_matrix = np.zeros((n, n))

for i, c1 in enumerate(chapters):
    for j, c2 in enumerate(chapters):
        shared = len(seen_hadm[c1] & seen_hadm[c2])
        union = len(seen_hadm[c1] | seen_hadm[c2])
        overlap_matrix[i, j] = shared / union if union > 0 else 0

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(overlap_matrix, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
#plt.colorbar(im, ax=ax, shrink=0.5, label="Jaccard similarity")
ax.set_xticks(range(n))
ax.set_xticklabels([f"{c} ({len(seen_hadm[c])})" for c in chapters], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(n))
ax.set_yticklabels([f"{c} ({len(seen_hadm[c])})" for c in chapters], fontsize=8)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{overlap_matrix[i,j]:.2f}", ha="center", va="center", fontsize=6)

ax.set_title("Admission overlap between chapters (Jaccard similarity)", fontsize=11, fontweight="bold")
ax.grid(False)
plt.tight_layout()
plt.show()


## compare with mimic top 100 features

In [ ]:
import pickle
with open ("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/data/top_features/mimic_top100_features.pkl", "rb") as f:
    global_top_100 = set(pickle.load(f))
global_top_100 = set(int(i) for i in global_top_100)

In [ ]:
# top features per chapter
n_top = 100
chapter_top_features = {chapter: set(sorted(prevalence[chapter], key=prevalence[chapter].get, reverse=True)[:n_top]) for chapter in all_chapters}


# plot prevalence only for top features across chapters
filtered_prevalence = {}
for chapter in chapter_top_features:
    filtered_prevalence[chapter] = {}

    top_itemids = chapter_top_features[chapter]

    for itemid in top_itemids:
        if itemid in prevalence[chapter]:
            filtered_prevalence[chapter][itemid] = prevalence[chapter][itemid]
     
# plot prevalence only for high prevalence
min_prevalence = 1
threshold_prevalence = {}
for chapter in all_chapters:
    threshold_prevalence[chapter] = {
        itemid: prev
        for itemid, prev in prevalence[chapter].items()
        if prev >= min_prevalence
    }






In [ ]:
# for all the itemids we have the prelavence even very small the filtered prevalence only include the itemids which are among the top n itemids within each chapter
# so the rest of the itemids are zero because they are not among top n but in prevalnce we also have the values for those

# all global and chapter itemids (100 top for each chapter + 100 global features)
all_itemids_to_plot = global_top_100.copy()
for chapter_set in chapter_top_features.values(): all_itemids_to_plot.update(chapter_set)
#print(len(all_itemids_to_plot))

prevalence_to_plot = threshold_prevalence # prevalence, filtered_prevalence, threshhold_prevalence

rows = []
for itemid in all_itemids_to_plot:
    for chapter in all_chapters:
        prev = prevalence_to_plot[chapter].get(itemid, 0) # prevalence, filtered_prevalence, threshhold_prevalence
        rows.append({
            "itemid": itemid,
            "label": itemid_label.get(itemid, str(itemid)),
            "chapter": chapter,
            "prevalence": prev,
            "is_global": itemid in global_top_100,
        })

bubble_df = pd.DataFrame(rows)

itemid_order_ids = (
    bubble_df.groupby("itemid")["prevalence"]
    .max()
    .sort_values(ascending=True)
    .index.tolist()
)
itemid_order_labels = [itemid_label.get(i, str(i)) for i in itemid_order_ids]

bubble_indexed = bubble_df.set_index(["itemid", "chapter"])

fig, ax = plt.subplots(figsize=(16, max(12, len(itemid_order_ids) * 0.3)))

for i, itemid in enumerate(itemid_order_ids):
    for j, chapter in enumerate(all_chapters):
        try:
            row = bubble_indexed.loc[(itemid, chapter)]
        except KeyError:
            continue
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        if row["prevalence"] == 0:
            continue
        color = "#2166ac" if row["is_global"] else "#d73027"
        ax.scatter(j, i, s=row["prevalence"] * 50, color=color, alpha=0.7)

ax.set_xticks(range(len(all_chapters)))
#ax.set_xticklabels(all_chapters, rotation=45, ha="right", fontsize=8)
ax.set_xticklabels([f"{chapter} (n={chapter_size[chapter]})" for chapter in bubble_df.chapter.unique()],fontsize=9,rotation=45, ha="right", )
ax.set_yticks(range(len(itemid_order_labels)))
ax.set_yticklabels(itemid_order_labels, fontsize=7)
ax.scatter([], [], s=100, color="#2166ac", alpha=0.7, label="global top 100")
ax.scatter([], [], s=100, color="#d73027", alpha=0.7, label="chapter-specific")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_title("Global + chapter-specific features — sorted by max prevalence", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

